In [1]:
import pandas as pd
import re
import json
from collections import Counter, defaultdict
from pathlib import Path
import sys

# ── CONFIG ──────────────────────────────────────────────────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

NOISE_PATTERNS = {
    "html_xml_tags":        r"<[^>]+>",
    "urls":                 r"https?://\S+|www\.\S+",
    "emails":               r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",
    "citations_brackets":   r"\[\d+\]|\[\d+,\s*\d+\]|\[[\w\s,]+\d{4}[\w\s,]*\]",
    "citations_parens":     r"\(\w[\w\s,\.]*\d{4}[a-z]?\)",
    "et_al":                r"\bet\s+al\.?",
    "standalone_numbers":   r"(?<!\w)\d+(?:\.\d+)?(?!\w)",
    "special_symbols":      r"[©®™°•·▪▸►◄●○□■×÷±≠≤≥≈∞∑∏√∂∆∫]",
    "non_ascii":            r"[^\x00-\x7F]+",
    "repeated_punctuation": r"[!?.,;:]{2,}",
    "bullet_artifacts":     r"^\s*[-•*·▪▸►]\s+",
    "page_markers":         r"(?i)\bpage\s+\d+\b|\bp\.\s*\d+\b",
    "figure_table_refs":    r"(?i)\b(fig(?:ure)?|table|appendix|section|chapter)\s*\.?\s*\d+",
    "excessive_whitespace": r"[ \t]{2,}|\n{3,}|\r",
    "non_breaking_spaces":  r"\xa0|\u00a0|\u2003|\u2002|\u200b",
    "lone_single_chars":    r"(?<!\w)[b-df-hj-np-tv-z](?!\w)",  # non-vowel single chars
    "header_like_caps":     r"\b[A-Z]{4,}\b",
    "trailing_leading_ws":  r"^\s+|\s+$",
}


def analyse_column(series: pd.Series, col_name: str) -> dict:
    """
    For each noise pattern, count:
    - how many rows it appears in
    - total match count across all rows
    - up to 5 example snippets
    """
    results = {}
    n_rows = len(series.dropna())

    for noise_type, pattern in NOISE_PATTERNS.items():
        row_hits = 0
        total_matches = 0
        examples = []

        for cell in series.dropna().astype(str):
            matches = re.findall(pattern, cell, flags=re.MULTILINE)
            if matches:
                row_hits += 1
                total_matches += len(matches)
                if len(examples) < 5:
                    examples.extend(
                        [m.strip()[:80] for m in matches if m.strip()][:3]
                    )

        if row_hits > 0:
            results[noise_type] = {
                "rows_affected": row_hits,
                "pct_rows": round(row_hits / n_rows * 100, 1),
                "total_occurrences": total_matches,
                "examples": list(dict.fromkeys(examples))[:5],  # dedupe
            }

    return results


def detect_boilerplate(series: pd.Series, top_n: int = 10) -> list:
    """
    Find repeated sentence-level fragments across rows.
    Returns top_n most common sentences (>=6 words) that appear in >1 row.
    """
    sentence_counter = Counter()

    for cell in series.dropna().astype(str):
        sentences = re.split(r"(?<=[.!?])\s+", cell)
        for sent in sentences:
            sent = sent.strip()
            word_count = len(sent.split())
            if word_count >= 6:
                sentence_counter[sent[:120]] += 1

    return [
        {"text": text, "occurrences": count}
        for text, count in sentence_counter.most_common(top_n)
        if count > 1
    ]


def compute_overall_stats(series: pd.Series) -> dict:
    """Basic stats about the text column."""
    lengths = series.dropna().astype(str).str.len()
    word_counts = series.dropna().astype(str).str.split().str.len()
    return {
        "total_rows": len(series),
        "null_rows": int(series.isna().sum()),
        "empty_string_rows": int((series.astype(str).str.strip() == "").sum()),
        "avg_char_length": round(lengths.mean(), 1),
        "min_char_length": int(lengths.min()),
        "max_char_length": int(lengths.max()),
        "avg_word_count": round(word_counts.mean(), 1),
    }


def run(filepath: str):
    path = Path(filepath)

    if path.suffix in (".xlsx", ".xls"):
        # openpyxl handles encoding internally — no charset issue here
        df = pd.read_excel(filepath, engine="openpyxl")
    elif path.suffix == ".csv":
        # Try UTF-8 first, fall back to Windows-1252 (CP1252) which covers
        # 0x80–0x9F smart-quotes, em-dashes etc. that break strict UTF-8
        for enc in ("utf-8", "cp1252", "latin-1"):
            try:
                df = pd.read_csv(filepath, encoding=enc)
                print(f"  [INFO] CSV read with encoding: {enc}")
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError("Could not decode CSV with utf-8, cp1252, or latin-1")
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    print(f"\n{'='*60}")
    print(f"  NLP NOISE ANALYSIS REPORT")
    print(f"  File : {path.name}")
    print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"{'='*60}\n")

    report = {}

    for col in TEXT_COLS:
        if col not in df.columns:
            print(f"  [SKIP] Column '{col}' not found in file.\n")
            continue

        print(f"\n{'─'*60}")
        print(f"  COLUMN: {col}")
        print(f"{'─'*60}")

        stats = compute_overall_stats(df[col])
        print(f"\n  📊 Basic Stats:")
        for k, v in stats.items():
            print(f"     {k:<25} {v}")

        noise = analyse_column(df[col], col)

        print(f"\n  🔍 Noise Patterns Found: {len(noise)}")
        if noise:
            sorted_noise = sorted(noise.items(), key=lambda x: -x[1]["rows_affected"])
            for noise_type, data in sorted_noise:
                print(f"\n     [{noise_type}]")
                print(f"       Rows affected : {data['rows_affected']} ({data['pct_rows']}%)")
                print(f"       Total hits    : {data['total_occurrences']}")
                if data["examples"]:
                    print(f"       Examples      : {data['examples'][:3]}")
        else:
            print("     ✅ No noise patterns detected.")

        boilerplate = detect_boilerplate(df[col])
        print(f"\n  🔁 Repeated Boilerplate Sentences (>1 occurrence):")
        if boilerplate:
            for item in boilerplate[:5]:
                print(f"     ({item['occurrences']}x) \"{item['text'][:90]}...\"")
        else:
            print("     ✅ No boilerplate detected.")

        report[col] = {
            "stats": stats,
            "noise_patterns": noise,
            "boilerplate": boilerplate,
        }

    # Save JSON report
    out_path = path.parent / f"{path.stem}_noise_report.json"
    with open(out_path, "w", encoding="utf-8", errors="replace") as f:
        json.dump(report, f, indent=2, ensure_ascii=True)

    print(f"\n\n{'='*60}")
    print(f"  ✅ JSON report saved → {out_path.name}")

if __name__ == "__main__":
    input_file=input("Enter path to CSV/XLSX file: ").strip()
    run(input_file)

  [INFO] CSV read with encoding: latin-1

  NLP NOISE ANALYSIS REPORT
  File : Copy of indian_budget_dataset.csv
  Shape: 670 rows × 6 cols


────────────────────────────────────────────────────────────
  COLUMN: chunk_text
────────────────────────────────────────────────────────────

  📊 Basic Stats:
     total_rows                670
     null_rows                 0
     empty_string_rows         0
     avg_char_length           2105.6
     min_char_length           52
     max_char_length           2668
     avg_word_count            339.4

  🔍 Noise Patterns Found: 11

     [standalone_numbers]
       Rows affected : 659 (98.4%)
       Total hits    : 13783
       Examples      : ['1', '2010', '11']

     [non_ascii]
       Rows affected : 439 (65.5%)
       Total hits    : 1865
       Examples      : ['\x92', '\x9d', '\x95']

     [header_like_caps]
       Rows affected : 427 (63.7%)
       Total hits    : 1441
       Examples      : ['RKVY', 'NABARD']

     [lone_single_chars]
  

In [2]:
import pandas as pd
import re
import json
from collections import Counter, defaultdict
from pathlib import Path
import sys

In [19]:
import pandas as pd
import re
import json
from pathlib import Path
import sys

TEXT_COLS = ["chunk_text", "summary_text"]

# Characters we know exist from the noise report
NON_ASCII_PATTERN   = r"[^\x00-\x7F]+"
SPECIAL_SYM_PATTERN = r"[©®™°•·▪▸►◄●○□■×÷±≠≤≥≈∞∑∏√∂∆∫]"

# CP1252 control chars that sneak in as "non-ASCII" in budget docs
CP1252_MAP = {
    "\u0092": "RIGHT SINGLE QUOTATION MARK (CP1252 0x92)",
    "\u009d": "RIGHT DOUBLE QUOTEMARK (CP1252 0x9D)",
    "\u0095": "BULLET (CP1252 0x95)",
    "\u0096": "EN DASH (CP1252 0x96)",
    "\u0097": "EM DASH (CP1252 0x97)",
    "\u0093": "LEFT DOUBLE QUOTATION MARK (CP1252 0x93)",
    "\u0094": "RIGHT DOUBLE QUOTATION MARK (CP1252 0x94)",
    "\u0091": "LEFT SINGLE QUOTATION MARK (CP1252 0x91)",
}


def describe_char(ch: str) -> str:
    """Return a human-readable label for any suspicious character."""
    if ch in CP1252_MAP:
        return CP1252_MAP[ch]
    cp = ord(ch)
    try:
        import unicodedata
        name = unicodedata.name(ch, "UNKNOWN")
    except Exception:
        name = "UNKNOWN"
    return f"U+{cp:04X} {name}"


def find_noise_chars(text: str) -> list[dict]:
    """
    Find all non-ASCII / special-symbol characters in text.
    Returns a list of {char, description, count, context} dicts.
    """
    from collections import Counter
    hits = Counter()
    for ch in text:
        if ord(ch) > 127:
            hits[ch] += 1

    results = []
    for ch, count in hits.most_common():
        # grab a small context window around first occurrence
        idx = text.find(ch)
        snippet = text[max(0, idx-30):idx+31].replace("\n", " ").strip()
        results.append({
            "char":        repr(ch),
            "description": describe_char(ch),
            "count":       count,
            "context":     f"...{snippet}...",
        })
    return results


def run(filepath: str):
    path = Path(filepath)

    # ── Read file ─────────────────────────────────────────────────────────
    if path.suffix in (".xlsx", ".xls"):
        df = pd.read_excel(filepath, engine="openpyxl")
    elif path.suffix == ".csv":
        for enc in ("utf-8", "cp1252", "latin-1"):
            try:
                df = pd.read_csv(filepath, encoding=enc)
                print(f"[INFO] Read with encoding: {enc}")
                break
            except UnicodeDecodeError:
                continue
    else:
        raise ValueError(f"Unsupported: {path.suffix}")

    inspection_rows = []  # will be saved to Excel

    for col in TEXT_COLS:
        if col not in df.columns:
            print(f"[SKIP] Column '{col}' not in file")
            continue

        print(f"\n{'='*60}")
        print(f"  COLUMN: {col}")
        print(f"{'='*60}")

        flagged = df[df[col].astype(str).str.contains(NON_ASCII_PATTERN, regex=True, na=False)]
        print(f"  Rows with non-ASCII chars: {len(flagged)} / {len(df)}\n")

        for _, row in flagged.iterrows():
            text = str(row[col])
            noise_chars = find_noise_chars(text)

            # console output
            id_col = "chunk_id" if "chunk_id" in df.columns else df.columns[0]
            print(f"  Row {row.get(id_col, '?')} | {col}")
            for nc in noise_chars:
                print(f"    {nc['char']:12} | {nc['description']:50} | x{nc['count']} | {nc['context'][:60]}")

            # collect for Excel export
            for nc in noise_chars:
                inspection_rows.append({
                    "chunk_id":    row.get("chunk_id", ""),
                    "document_id": row.get("document_id", ""),
                    "year":        row.get("year", ""),
                    "column":      col,
                    "char":        nc["char"],
                    "description": nc["description"],
                    "count_in_row":nc["count"],
                    "context":     nc["context"],
                    "full_text":   text[:300] + ("..." if len(text) > 300 else ""),
                })

    # ── Save inspection Excel ─────────────────────────────────────────────
    if inspection_rows:
        out_df   = pd.DataFrame(inspection_rows)
        out_path = path.parent / f"{path.stem}_nonascii_inspection.xlsx"
        out_df.to_excel(out_path, index=False)
        print(f"\n✅ Saved inspection sheet → {out_path}")
        print(f"   {len(out_df)} rows written (one row per unique char per chunk)")
    else:
        print("\n✅ No non-ASCII characters found.")


if __name__ == "__main__":
    input_file="Copy of indian_budget_summarization_dataset.csv"
    run(input_file)

[INFO] Read with encoding: utf-8

  COLUMN: chunk_text
  Rows with non-ASCII chars: 113 / 159

  Row UB_2022_23_c1 | chunk_text
    '’'          | U+2019 RIGHT SINGLE QUOTATION MARK                 | x6 | ...f Finance February 1, 2022 Hon’ble Speaker, I present the
    '\uf0b7'     | U+F0B7 UNKNOWN                                     | x3 | ...attain the vision. They are:  Complementing the macro-ec
    '–'          | U+2013 EN DASH                                     | x2 | ...system for the middle classes – a vast and wide section w
  Row UB_2022_23_c2 | chunk_text
    '’'          | U+2019 RIGHT SINGLE QUOTATION MARK                 | x3 | ...s the result of our government’s strong commitment for ‘m
    '‘'          | U+2018 LEFT SINGLE QUOTATION MARK                  | x2 | ...nment’s strong commitment for ‘minimum government & maxim
  Row UB_2022_23_c3 | chunk_text
    '’'          | U+2019 RIGHT SINGLE QUOTATION MARK                 | x3 | ...will help realize the country’s econ

In [8]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: C:\Users\gauth\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import re
import unicodedata

# ── CELL 1: Load your file ────────────────────────────────────────────────────
for enc in ("utf-8", "cp1252", "latin-1"):
    try:
        df = pd.read_csv("Copy of indian_budget_summarization_dataset.csv", encoding=enc)
        print(f"Read with encoding: {enc}")
        break
    except UnicodeDecodeError:
        continue

# If xlsx:
# df = pd.read_excel("your_file.xlsx", engine="openpyxl")

df.shape

Read with encoding: utf-8


(159, 6)

In [22]:
# ── CELL 2: Define cleaning functions ────────────────────────────────────────

# CP1252 replacement map — each Windows-1252 control char → clean ASCII equiv
CP1252_REPLACEMENTS = {
    "\u0091": "'",   # left single quote
    "\u0092": "'",   # right single quote  ← your 0x92
    "\u0093": '"',   # left double quote
    "\u0094": '"',   # right double quote   # bullet              ← your 0x95
    "\u0095": " ",   # bullet              ← your 0x95
    "\u0096": "-",   # en dash
    "\u0097": "--",  # em dash
    "\u009d": '"',   # right double quote variant ← your 0x9D
    "\u00ae": "",    # ® registered mark
    "\u00b1": "+/-", # ±
}

SPECIAL_SYMBOLS = r"[©®™°•·▪▸►◄●○□■×÷±≠≤≥≈∞∑∏√∂∆∫]"


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Step 1: Replace known CP1252 artifacts with ASCII equivalents
    for bad_char, replacement in CP1252_REPLACEMENTS.items():
        text = text.replace(bad_char, replacement)

    # Step 2: Normalize unicode — converts accented chars like é → e
    # NFKD decomposes, then encode/decode drops the combining diacritics
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")

    # Step 3: Remove leftover special symbols
    text = re.sub(SPECIAL_SYMBOLS, "", text)

    # Step 4: Collapse excessive whitespace
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [23]:

# ── CELL 3: Preview before vs after on 3 affected rows ───────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

for col in TEXT_COLS:
    mask = df[col].astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False)
    sample = df[mask][col].head(3)
    print(f"\n{'='*60}\n  {col} — {mask.sum()} rows affected\n{'='*60}")
    for idx, val in sample.items():
        cleaned = clean_text(val)
        print(f"\n  [ROW {idx}] BEFORE: {repr(val[:120])}")
        print(f"           AFTER : {repr(cleaned[:120])}")



  chunk_text — 113 rows affected

  [ROW 0] BEFORE: 'Budget 2022-2023 Speech of Nirmala Sitharaman Minister of Finance February 1, 2022 Hon’ble Speaker, I present the Budget'
           AFTER : 'Budget 2022-2023 Speech of Nirmala Sitharaman Minister of Finance February 1, 2022 Honble Speaker, I present the Budget '

  [ROW 1] BEFORE: 'Taking forward this agenda, and to mark 75 years of our independence, it is proposed to set up 75 Digital Banking Units '
           AFTER : 'Taking forward this agenda, and to mark 75 years of our independence, it is proposed to set up 75 Digital Banking Units '

  [ROW 2] BEFORE: 'To prepare for this, orderly urban development is of critical importance. This will help realize the country’s economic '
           AFTER : 'To prepare for this, orderly urban development is of critical importance. This will help realize the countrys economic p'

  summary_text — 75 rows affected

  [ROW 0] BEFORE: 'The 2022–23 Union Budget speech highlighted India’s strong 

In [24]:
# ── CELL 4: Apply cleaning to both columns ────────────────────────────────────
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)
        print(f"Cleaned: {col}")


Cleaned: chunk_text
Cleaned: summary_text


In [25]:
# ── CELL 5: Verify — should print 0 for both ─────────────────────────────────
for col in TEXT_COLS:
    remaining = df[col].astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum()
    print(f"{col}: {remaining} rows still have non-ASCII chars")

chunk_text: 0 rows still have non-ASCII chars
summary_text: 0 rows still have non-ASCII chars


In [26]:
EXACT_REPLACEMENTS = {
    "http://indiabudget.nic.in" : "",
    "`"                         : "Rs.",
    "$"                         : "dollars",
}

def apply_replacements(text: str) -> str:
    if not isinstance(text, str):
        return text
 
    for find, replace in EXACT_REPLACEMENTS.items():
        text = text.replace(find, replace)

    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()
 
 
TEXT_COLS = ["chunk_text", "summary_text"]
 
for col in TEXT_COLS:
    if col not in df.columns:
        continue
    print(f"\n{'='*60}\n  {col}\n{'='*60}")
    for idx, row in df[[col]].head(3).iterrows():
        original = str(row[col])
        cleaned  = apply_replacements(original)
        if original != cleaned:
            print(f"\n  [Row {idx}]")
            print(f"  BEFORE: {original[:200]}")
            print(f"  AFTER : {cleaned[:200]}")
 

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(apply_replacements)
        print(f"Done: {col}")


  chunk_text

  [Row 2]
  BEFORE: To prepare for this, orderly urban development is of critical importance. This will help realize the countrys economic potential, including livelihood opportunities for the demographic dividend. For t
  AFTER : To prepare for this, orderly urban development is of critical importance. This will help realize the countrys economic potential, including livelihood opportunities for the demographic dividend. For t

  summary_text
Done: chunk_text
Done: summary_text


In [ ]:
# ── CELL 6: Save ─────────────────────────────────────────────────────────────
df.to_csv("your_file_cleaned_1.csv", index=False, encoding="utf-8")
# df.to_excel("your_file_cleaned.xlsx", index=False)
print("Saved.")

Saved.


In [31]:
df_dummy= df.drop(["chunk_id", "document_id", "year", "chunk_filename"], axis=1)
df_dummy.to_csv("dummy file.csv", index=False, encoding="utf-8")

In [ ]:
# ── CELL 1: Patterns to extract Yojana / Mission / Abhiyan terms ─────────────
import re
import json
from collections import Counter

# Anchor keywords that signal a Hindi scheme name
HINDI_ANCHORS = [
    "Yojana", "Yojna", "Abhiyan", "Mission", "Vikas", "Krishi",
    "Grameen", "Sadak", "Nirman", "Sansad", "Mantri", "Pradhan",
    "Rashtriya", "Indira", "Rajiv", "Mahatma", "Bharat", "Adarsh",
    "Gram", "Jan", "Dhan", "Suraksha", "Bima", "Panchayat",
    "Vidyutikaran", "Awas", "Swachh", "Ujjwala", "Fasal", "Sinchai",
    "Jal", "Swasthya", "Jeevan", "Urja", "Sahaj", "Ayushman",
    "Poshan", "Antyodaya", "Kaushal", "Gramin", "Deendayal",
    "Annapurna", "Saubhagya", "Atmanirbhar", "AtmaNirbhar",
]

anchor_group = "|".join(re.escape(a) for a in HINDI_ANCHORS)

# Match a capitalized multi-word phrase that contains at least one anchor word
# Allows 1–8 title-cased or all-caps words before/after the anchor
SCHEME_PATTERN = re.compile(
    r'\b(?:[A-Z][a-zA-Z]+\s+){0,7}(?:' + anchor_group + r')(?:\s+[A-Z][a-zA-Z]+){0,5}\b'
)


# ── CELL 2: Extract from both text columns ────────────────────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

raw_hits = Counter()

for col in TEXT_COLS:
    if col not in df.columns:
        continue
    for cell in df[col].dropna().astype(str):
        matches = SCHEME_PATTERN.findall(cell)
        for m in matches:
            cleaned = m.strip()
            # filter out noise — must be at least 2 words, not all stopwords
            words = cleaned.split()
            if len(words) >= 2:
                raw_hits[cleaned] += 1

print(f"Unique raw candidates : {len(raw_hits)}")
print(f"Total occurrences     : {sum(raw_hits.values())}\n")

print("Top 50 candidates:")
for term, count in raw_hits.most_common(50):
    print(f"  ({count:3}x)  {term}")


# ── CELL 3: Deduplicate — collapse spelling variants into canonical forms ──────
# Strategy: if term A is a substring of term B, keep B (the longer, fuller name)
# Also manually map known misspellings → correct form

VARIANT_MAP = {
    # misspellings / shortened forms → canonical
    "Pradhan Mantri Gram Sadak Yojna"         : "Pradhan Mantri Gram Sadak Yojana",
    "Pradhan Mantri Krishi Sinchayee Yojana"  : "Pradhan Mantri Krishi Sinchai Yojana",
    "Prdhan Mantri Krishi Sinchai Yojana"     : "Pradhan Mantri Krishi Sinchai Yojana",
    "Pradhanmantri Gram Sinchai Yojana"       : "Pradhan Mantri Krishi Sinchai Yojana",
    "Pradhan Mantri Krishi Sinchaii Yojna"    : "Pradhan Mantri Krishi Sinchai Yojana",
    "Krishi Sinchai Yojana"                   : "Pradhan Mantri Krishi Sinchai Yojana",
    "Krishi Sinchayee Yojana"                 : "Pradhan Mantri Krishi Sinchai Yojana",
    "Pradhan Mantri Awas Yojna"               : "Pradhan Mantri Awas Yojana",
    "PM Awas Yojana"                          : "Pradhan Mantri Awas Yojana",
    "Pradhan Mantri Awaas Yojana"             : "Pradhan Mantri Awas Yojana",
    "PM Gram Sadak Yojana"                    : "Pradhan Mantri Gram Sadak Yojana",
    "Prime Minister Gram Sadak Yojana"        : "Pradhan Mantri Gram Sadak Yojana",
    "Rural Development Pradhan Mantri Gram Sadak Yojana": "Pradhan Mantri Gram Sadak Yojana",
    "Pradhan Mantri Suraksha Bima Yojna"      : "Pradhan Mantri Suraksha Bima Yojana",
    "PM Suraksha Bima and PM Jeevan Jyoti Yojana": "Pradhan Mantri Suraksha Bima Yojana",
    "Pradhan Mantri Jeevan Jyoti Beema Yojana": "Pradhan Mantri Jeevan Jyoti Bima Yojana",
    "Pradhan Mantri Kaushal Vikas Yojna"      : "Pradhan Mantri Kaushal Vikas Yojana",
    "PM Jan Dhan Yojana"                      : "Pradhan Mantri Jan Dhan Yojana",
    "Prime Minister Jan Dhan Yojana"          : "Pradhan Mantri Jan Dhan Yojana",
    "PM Jan Dhan"                             : "Pradhan Mantri Jan Dhan Yojana",
    "Indira Awas Yojna"                       : "Indira Awas Yojana",
    "Rajiv Gandhi Grameen Vidyutikaran Yojna" : "Rajiv Gandhi Grameen Vidyutikaran Yojana",
    "Swachh Bharat Abhiyan"                   : "Swachh Bharat Mission",
    "Swachch Bharat Mission"                  : "Swachh Bharat Mission",
    "Swachch Bharat"                          : "Swachh Bharat Mission",
    "Urban Swachh Bharat Mission"             : "Swachh Bharat Mission",
    "Sanitation Swachh Bharat Mission"        : "Swachh Bharat Mission",
    "Rashtriya Swasthya Bima Yojana"          : "Rashtriya Swasthya Bima Yojana",
    "Rashtriya Swasthiya Bima Yojana"         : "Rashtriya Swasthya Bima Yojana",
    "Present Rashtriya Swasthya Bima Yojana"  : "Rashtriya Swasthya Bima Yojana",
    "Deendayal Upadhayaya Gram Jyoti Yojna"   : "Deen Dayal Upadhyaya Gram Jyoti Yojana",
    "Deendayal Upadhyaya Gram Jyoti Yojana"   : "Deen Dayal Upadhyaya Gram Jyoti Yojana",
    "Ministry of Power Deen Dayal Upadhyaya Gram Jyoti Yojana": "Deen Dayal Upadhyaya Gram Jyoti Yojana",
    "Parmparagat Krishi Vikas Yojana"         : "Paramparagat Krishi Vikas Yojana",
    "Dhaanya Krishi Yojana"                   : "Dhaanya Krishi Yojana",
    "Varishta Bima Yojana"                    : "Varishtha Pension Bima Yojana",
    "Varishtha Pension Bima Yojna"            : "Varishtha Pension Bima Yojana",
    "Atma Nirbhar Bharat"                     : "AtmaNirbhar Bharat",
    "Atmanirbhar Bharat"                      : "AtmaNirbhar Bharat",
    "PM Fasal Bima Yojana"                    : "Pradhan Mantri Fasal Bima Yojana",
    "Prime Minister Fasal Bima Yojana"        : "Pradhan Mantri Fasal Bima Yojana",
    "PM Jan Arogya Yojana"                    : "Pradhan Mantri Jan Arogya Yojana",
    "Prime Minister Jan Arogya Yojana"        : "Pradhan Mantri Jan Arogya Yojana",
    "PM Matsya Sampada Yojana"                : "Pradhan Mantri Matsya Sampada Yojana",
    "Sampada Yojana"                          : "Pradhan Mantri Krishi Sampada Yojana",
    "Sab ka Saath Sab ka Vikas"               : "Sabka Saath Sabka Vikas",
    "Pradhan Mantri Swasthya Suraksha Yojana" : "Pradhan Mantri Swasthya Suraksha Yojana",
    "Pardhan Mantri Swasthya Suraksha Yojana" : "Pradhan Mantri Swasthya Suraksha Yojana",
    "Family Welfare Pradhan Mantri Swasthya Suraksha Yojana": "Pradhan Mantri Swasthya Suraksha Yojana",
    "Deen Dayal Upadhyay Grameen Kaushalya Yojana": "Deen Dayal Upadhyaya Grameen Kaushalya Yojana",
    "Deen Dayal Upadhyay Gramin Kaushal Yojana": "Deen Dayal Upadhyaya Grameen Kaushalya Yojana",
    "Deendayal Antyodaya Yojana National Rural Livelihood Mission": "Deen Dayal Antyodaya Yojana",
}

def canonicalize(term: str) -> str:
    return VARIANT_MAP.get(term, term)

# Build canonical counter
canonical_hits = Counter()
for term, count in raw_hits.items():
    canonical_hits[canonicalize(term)] += count

# Remove terms that are substrings of a longer term in the same set
# e.g. "Pradhan Mantri" should not survive if "Pradhan Mantri Jan Dhan Yojana" exists
all_terms = list(canonical_hits.keys())
filtered_terms = []
for term in all_terms:
    is_substring = any(
        term != other and term in other
        for other in all_terms
    )
    if not is_substring:
        filtered_terms.append(term)

canonical_hits = Counter({t: canonical_hits[t] for t in filtered_terms})

print(f"\nAfter deduplication: {len(canonical_hits)} unique canonical terms\n")
for term, count in canonical_hits.most_common():
    print(f"  ({count:3}x)  {term}")


# ── CELL 4: Build and save the glossary JSON ──────────────────────────────────
# Descriptions are placeholders — fill in or enrich from domain_glossary.json
# If domain_glossary.json already exists, auto-fill descriptions from it

try:
    with open("domain_glossary.json", encoding="utf-8") as f:
        existing_glossary = json.load(f)
except FileNotFoundError:
    existing_glossary = {}

new_glossary = {}
missing = []  # terms we found but have no description for yet

for term, _ in canonical_hits.most_common():
    if term in existing_glossary:
        new_glossary[term] = existing_glossary[term]
    else:
        new_glossary[term] = ""  # blank — needs manual filling
        missing.append(term)

with open("domain_glossary_extracted.json", "w", encoding="utf-8") as f:
    json.dump(new_glossary, f, indent=2, ensure_ascii=False)

print(f"\nSaved domain_glossary_extracted.json")
print(f"  Filled from existing glossary : {len(new_glossary) - len(missing)}")
print(f"  Needs description added       : {len(missing)}")
if missing:
    print("\n  Terms needing descriptions:")
    for t in missing:
        print(f"    - {t}")

In [3]:
df=pd.read_csv("indian_budget_dataset_cleaned.csv", encoding="utf-8")

In [27]:
# ── CELL: Remove ordered / pointed list markers ──────────────────────────────
import re

# Matches at start of a token/line or after whitespace:
# i) ii) iii) iv) v) ... (roman numerals)
# 1) 2) 10) ... (numeric)
# a) b) c) ... (alpha)
# i. ii. 1. a. (dot variant)
# Also handles (i) (1) (a) wrapped in parens

LIST_MARKER = re.compile(
    r'(?<!\w)'                          # not preceded by a word char
    r'(?:'
        r'\(?\b(?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2}|xi{0,3})\b\)?'  # roman: i ii iii iv ... xiii
        r'|'
        r'\(?\b[a-zA-Z]\b\)?'           # single alpha: a) b) (a) (b)
        r'|'
        r'\(?\b\d{1,3}\b\)?'            # numeric: 1) 2) (1) (2) up to 999
    r')'
    r'(?=\s*[.)]\s|\s)',                # must be followed by . or ) then space, OR just space
    re.IGNORECASE
)

PLACEHOLDER = "\x00"  # safe byte, won't appear in normal text

def remove_list_markers(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Step 0: protect known abbreviations before any regex runs
    text = text.replace("i.e.", f"{PLACEHOLDER}ie{PLACEHOLDER}")
    text = text.replace("e.g.", f"{PLACEHOLDER}eg{PLACEHOLDER}")
    text = text.replace("etc.", f"{PLACEHOLDER}etc{PLACEHOLDER}")

    # Pass 1 — wrapped (i) (ii) (a) (1) (2) — all included
    text = re.sub(
        r'(?:^|(?<=\s))'
        r'\((?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2}|xi{0,3}|[a-zA-Z]|\d{1,3})\)'
        r'\s*',
        '. ', text, flags=re.IGNORECASE
    )

    # Pass 2 — bare i. ii. a. i) a) 1. 2. 1) 2) — all included
    text = re.sub(
        r'(?:^|(?<=\s))'
        r'(?:i{1,3}|iv|v|vi{0,3}|ix|x{1,2}|xi{0,3}|[a-zA-Z]|\d{1,3})'
        r'[.)]\s+',
        '. ', text, flags=re.IGNORECASE
    )

    # Step 3: restore protected abbreviations
    text = text.replace(f"{PLACEHOLDER}ie{PLACEHOLDER}", "i.e.")
    text = text.replace(f"{PLACEHOLDER}eg{PLACEHOLDER}", "e.g.")
    text = text.replace(f"{PLACEHOLDER}etc{PLACEHOLDER}", "etc.")

    # Clean up duplicate fullstops and extra spaces
    text = re.sub(r'\.(\s*\.)+', '.', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


# ── Preview before / after on 5 rows ─────────────────────────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

for col in TEXT_COLS:
    if col not in df.columns:
        continue
    # Find rows that actually have markers
    mask = df[col].astype(str).str.contains(
        r'(?<!\w)(?:\(?\b(?:i{1,3}|iv|v|vi{0,3}|ix|x)\b[.)]\s|\b\d{1,2}[.)]\s|\b[a-z][.)]\s)',
        regex=True, flags=re.IGNORECASE, na=False
    )
    print(f"\n{col} — {mask.sum()} rows contain list markers")
    for idx, row in df[mask].head(3).iterrows():
        original = str(row[col])[:300]
        cleaned  = remove_list_markers(str(row[col]))[:300]
        print(f"\n  [Row {idx}]")
        print(f"  BEFORE: {original}")
        print(f"  AFTER : {cleaned}")


# ── Apply ─────────────────────────────────────────────────────────────────────
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(remove_list_markers)
        print(f"\nCleaned: {col}")


chunk_text — 120 rows contain list markers

  [Row 0]
  BEFORE: Budget 2022-2023 Speech of Nirmala Sitharaman Minister of Finance February 1, 2022 Honble Speaker, I present the Budget for the year 2022-23. At the outset, I want to take a moment to express my empathy for those who had to bear adverse health and economic effects of the pandemic. The overall, sharp
  AFTER : Budget 2022-2023 Speech of Nirmala Sitharaman Minister of Finance February 1, 2022 Honble Speaker, I present the Budget for the year 2022-23. At the outset, I want to take a moment to express my empathy for those who had to bear adverse health and economic effects of the pandemic. The overall, sharp

  [Row 1]
  BEFORE: Taking forward this agenda, and to mark 75 years of our independence, it is proposed to set up 75 Digital Banking Units (DBUs) in 75 districts of the country by Scheduled Commercial Banks. The financial support for digital payment ecosystem announced in the previous Budget will continue in 2022-23. 
 

In [29]:
# ── CELL 1: Define your replacements here ────────────────────────────────────
# Three types of rules — add as many entries as you need
# ─────────────────────────────────────────────────────────────────────────────

# TYPE 1: Exact string replacements  →  { "find": "replace_with" }
EXACT_REPLACEMENTS = {
          # just remove it
}

# TYPE 2: Regex replacements  →  { "pattern": "replace_with" }
# Use r"..." strings. Capture groups like \1 work in replace_with.
REGEX_REPLACEMENTS = [
    (r'(?<=\d)/-', ''),
]

# TYPE 3: Whole-word replacements  →  { "word": "replace_with" }
# Unlike EXACT, these only match full words — won't touch "NABARD" if you add "NAB"
WHOLE_WORD_REPLACEMENTS = {
}


# ── CELL 2: Replacement engine ────────────────────────────────────────────────
import re

def apply_replacements(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Type 1 — exact, in order
    for find, replace in EXACT_REPLACEMENTS.items():
        text = text.replace(find, replace)

    # Type 2 — regex, in order
    for pattern, replace in REGEX_REPLACEMENTS:
        text = re.sub(pattern, replace, text)

    # Type 3 — whole word, in order
    for word, replace in WHOLE_WORD_REPLACEMENTS.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', replace, text)

    # Final cleanup — collapse extra whitespace
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ── CELL 3: Preview on 3 rows before applying ─────────────────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

for col in TEXT_COLS:
    if col not in df.columns:
        continue
    print(f"\n{'='*60}\n  {col}\n{'='*60}")
    for idx, row in df[[col]].head(3).iterrows():
        original = str(row[col])
        cleaned  = apply_replacements(original)
        if original != cleaned:
            print(f"\n  [Row {idx}]")
            print(f"  BEFORE: {original[:200]}")
            print(f"  AFTER : {cleaned[:200]}")


# ── CELL 4: Apply to dataframe ────────────────────────────────────────────────
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(apply_replacements)
        print(f"Done: {col}")


  chunk_text

  summary_text
Done: chunk_text
Done: summary_text


In [31]:
output_filepath = "indian_budget_dataset_cleaned_training_sum.csv"

df.to_csv(output_filepath, index=False, encoding='utf-8')

print(f"🎉 Success! Cleaned file saved to: {output_filepath}")

🎉 Success! Cleaned file saved to: indian_budget_dataset_cleaned_training_sum.csv


In [30]:
# ── CELL 1: Honorific rules ───────────────────────────────────────────────────
import re

# Group A — pure ceremony/address phrases → remove entirely, nothing follows worth keeping
REMOVE_ENTIRELY = [
    r"Madam\s+Speaker[,.]?",
    r"Honourable\s+Speaker\s+Sir[,.]?",
    r"Honourable\s+Speaker[,.]?",
    r"Mr\.\s+Speaker\s+Sir[,.]?",   # must come before standalone Sir
    r"Speaker\s+Sir[,.]?",
    r"Hon'ble\s+Members?[,.]?",
    r"honourable\s+members?[,.]?",
    r"august\s+House[,.]?",
    r"Madam[,.]?(?=\s)",
    r"(?<!\w)Sir(?!\w)[,.]?",
]

# Group B — kept intentionally empty
# Shri, Dr., Mr., Smt., Justice etc. are preserved as-is
TITLE_BEFORE_NAME = []


# ── CELL 2: Cleaning function ─────────────────────────────────────────────────

def remove_honorifics(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Group A — remove entirely
    for pat in REMOVE_ENTIRELY:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)

    # Group B — strip title, capitalize the first letter of the following word
    # Regex captures the word immediately after the title
    for pat in TITLE_BEFORE_NAME:
        def _capitalize_next(m):
            # m.group(1) is the word right after the title
            word = m.group(1)
            return word[0].upper() + word[1:] if word else ''
        text = re.sub(
            pat + r'([A-Za-z]\w*)',
            _capitalize_next,
            text
        )

    # Cleanup
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ── CELL 3: Preview ───────────────────────────────────────────────────────────
samples = [
    "Madam Speaker, I present the budget to this august House.",
    "Hon'ble Members may be curious as to why this was done.",
    "A Committee under Dr. C. Rangarajan has been set up.",
    "Shri Sam Pitroda heads the National Innovation Council.",
    "Mr. Speaker Sir, the Budget for the year 2019-20.",
    "Strong support lent by UPA Chairperson Smt. Sonia Gandhi.",
    "With these words, Madam Speaker, I commend the Budget to the House.",
    "Justice B. N. Srikrishna chairs the commission.",
]

print(f"{'BEFORE':<65}  AFTER")
print("─" * 120)
for s in samples:
    print(f"{s:<65}  {remove_honorifics(s)}")


# ── CELL 4: Apply to dataframe ────────────────────────────────────────────────
TEXT_COLS = ["chunk_text", "summary_text"]

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(remove_honorifics)
        print(f"Done: {col}")

BEFORE                                                             AFTER
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Madam Speaker, I present the budget to this august House.          I present the budget to this
Hon'ble Members may be curious as to why this was done.            may be curious as to why this was done.
A Committee under Dr. C. Rangarajan has been set up.               A Committee under Dr. C. Rangarajan has been set up.
Shri Sam Pitroda heads the National Innovation Council.            Shri Sam Pitroda heads the National Innovation Council.
Mr. Speaker Sir, the Budget for the year 2019-20.                  the Budget for the year 2019-20.
Strong support lent by UPA Chairperson Smt. Sonia Gandhi.          Strong support lent by UPA Chairperson Smt. Sonia Gandhi.
With these words, Madam Speaker, I commend the Budget to the House.  With these words, I commend the Budget to the House.
Justice B. N.

In [46]:
output_filepath = "indian_budget_dataset_cleaned.csv"

df.to_csv(output_filepath, index=False, encoding='utf-8')

print(f"🎉 Success! Cleaned file saved to: {output_filepath}")

🎉 Success! Cleaned file saved to: indian_budget_dataset_cleaned.csv


In [32]:
df_training= pd.read_csv("indian_budget_dataset_cleaned_training_sum.csv", encoding="utf-8")
df_training= df_training.drop(["chunk_id", "document_id", "year", "chunk_filename"], axis=1)

In [33]:
df_training.to_csv("training_file.csv", index=False, encoding='utf-8')